# AI MetroFlow: Machine Learning Model Training & Evaluation
## Predictive Public Transit Intelligence Platform for Crowd Management and Schedule Optimization

**Objective**: Predict subway crowd congestion levels (`Low`, `Medium`, `High`) using NYC Subway traffic data to enable proactive schedule optimization and crowd management.

### Pipeline Stages:
1. **Data Validation & Dataset Summary**
2. **Target Leakage Investigation & Prevention**
3. **Feature Preprocessing & Encoding**
4. **Model Training & Benchmarking** (Decision Tree vs. Random Forest vs. XGBoost)
5. **Performance Evaluation** (Accuracy, Precision, Recall, F1, Confusion Matrix)
6. **Feature Importance & Operational Insights**
7. **Model Serialization & Inference Demonstration**

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Append src to path
sys.path.append(os.path.abspath("../src"))

from utils import (
    load_and_validate_data,
    preprocess_features,
    evaluate_model,
    plot_confusion_matrix,
    plot_feature_importances,
    save_model_bundle,
    load_model_bundle,
    get_congestion_recommendation,
)
from prediction import CongestionPredictor

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
%matplotlib inline

## 1. Data Validation & Summary
We load the engineered feature dataset `data/features.csv` and inspect its shape, schema, null counts, and class balance.

In [2]:
# Load a balanced sample of 150,000 rows for responsive, high-fidelity modeling
df, summary = load_and_validate_data(sample_size=150000, random_state=42)

print("=== Dataset Summary ===")
print(f"Total Rows Loaded:    {summary['rows']:,}")
print(f"Total Columns:        {summary['columns']}")
print(f"Missing Values:       {summary['total_missing_values']}")
print(f"Class Distribution:   {summary['class_counts']}")
print(f"Class Proportions:    {summary['class_proportions']}")

df.head(3).T

## 2. Target Leakage Review & Prevention

> **Why Total_Traffic, Entries, and Exits must NOT be model features:**
1. `Total_Traffic` is computed directly as `Entries + Exits`.
2. `Congestion_Level` is derived from the 33rd and 66th quantiles of `Total_Traffic`.
3. Including `Total_Traffic`, `Entries`, or `Exits` allows decision trees to trivially recover the quantile split, resulting in artificial ~99%+ accuracy that fails completely in real-world deployment when future passenger counts are unknown.

Below, we empirically verify this leakage difference.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

y = df["Congestion_Level"].values

# Model A: Clean Features (Temporal + Station only)
X_clean, _, cols_clean = preprocess_features(df, is_training=True, include_leakage_features=False)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clean, y, test_size=0.2, random_state=42, stratify=y)

rf_clean = RandomForestClassifier(n_estimators=40, max_depth=14, random_state=42, n_jobs=-1)
rf_clean.fit(X_train_c, y_train_c)
acc_clean = rf_clean.score(X_test_c, y_test_c)

# Model B: Leaked Features (Temporal + Station + Entries + Exits)
X_leaked, _, cols_leaked = preprocess_features(df, is_training=True, include_leakage_features=True)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaked, y, test_size=0.2, random_state=42, stratify=y)

rf_leaked = RandomForestClassifier(n_estimators=40, max_depth=14, random_state=42, n_jobs=-1)
rf_leaked.fit(X_train_l, y_train_l)
acc_leaked = rf_leaked.score(X_test_l, y_test_l)

print(f"Model A (Clean Features - Realistic): Accuracy = {acc_clean:.4f}")
print(f"Model B (With Entries & Exits - Leaked): Accuracy = {acc_leaked:.4f}")
print("\nVerified: Target leakage causes artificial 99%+ accuracy. Excluded from production model.")

## 3. Model Training & Comparison
We train and benchmark three distinct algorithms using stratified 80/20 train/test split on clean features:
- **Decision Tree Classifier** (Baseline)
- **Random Forest Classifier** (Ensemble Bagging)
- **XGBoost Classifier** (Gradient Boosting)

In [4]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# Prepare features and target
X, encoders, feature_cols = preprocess_features(df, is_training=True, include_leakage_features=False)
target_encoder = LabelEncoder()
y_enc = target_encoder.fit_transform(y)

X_train, X_test, y_train_enc, y_test_enc, y_train, y_test = train_test_split(
    X, y_enc, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=16, min_samples_split=20, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, max_depth=7, learning_rate=0.1, random_state=42, eval_metric="mlogloss", n_jobs=-1),
}

results = {}
fitted_models = {}

for name, clf in models.items():
    t0 = time.time()
    if name == "XGBoost":
        clf.fit(X_train, y_train_enc)
        y_pred = target_encoder.inverse_transform(clf.predict(X_test))
    else:
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
    t_elapsed = time.time() - t0
    
    metrics = evaluate_model(y_test, y_pred, model_name=name)
    metrics["train_time"] = round(t_elapsed, 2)
    results[name] = metrics
    fitted_models[name] = clf

## 4. Evaluation & Model Comparison Results
Comparative analysis of Accuracy, Weighted F1 Score, Precision, Recall, and Training Latency.

In [5]:
comparison_df = pd.DataFrame([
    {
        "Model": m["model_name"],
        "Accuracy": m["accuracy"],
        "F1 (Weighted)": m["f1_weighted"],
        "Precision": m["precision_weighted"],
        "Recall": m["recall_weighted"],
        "Train Time (s)": m["train_time"],
    }
    for m in results.values()
])

print(comparison_df.to_string(index=False))

best_model_name = max(results, key=lambda k: results[k]["f1_weighted"])
print(f"\n>>> Best Selected Model: {best_model_name} (F1 Score: {results[best_model_name]['f1_weighted']:.4f}) <<<")

## 5. Confusion Matrix & Classification Report (Best Model)

In [6]:
best_metrics = results[best_model_name]
print("=== Classification Report ===\n")
print(best_metrics["classification_report"])

cm = np.array(best_metrics["confusion_matrix"])
classes = ["Low", "Medium", "High"]

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title(f"Confusion Matrix - {best_model_name}", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Congestion Level")
plt.ylabel("Actual Congestion Level")
plt.tight_layout()
plt.show()

## 6. Feature Importance & Transit Insights
Identifying which factors most strongly predict congestion across the transit network.

In [7]:
best_clf = fitted_models[best_model_name]
importances = pd.Series(best_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
bars = plt.barh(importances.index[::-1], importances.values[::-1], color="#1f77b4", edgecolor="#0e466d")
plt.xlabel("Relative Importance Score")
plt.title(f"Feature Importance Ranking - {best_model_name}", fontsize=12, fontweight="bold")
plt.xlim(0, max(importances.values) * 1.15)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.003, bar.get_y() + bar.get_height() / 2, f"{width:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\n=== Top Contributing Features ===")
for i, (feat, score) in enumerate(importances.items(), start=1):
    print(f"{i:2d}. {feat:<22} : {score * 100:6.2f}%")

## 7. Real-Time Prediction & Recommendation Engine
Demonstrating end-to-end inference with `CongestionPredictor` on realistic transit scenarios.

In [8]:
predictor = CongestionPredictor()

# Scenario: Morning rush hour at Times Sq - 42 St
rush_scenario = {
    "Stop Name": "Times Sq - 42 St",
    "Borough": "M",
    "Structure": "Subway",
    "Latitude": 40.75529,
    "Longitude": -73.987495,
    "hour": 8,
    "day": 14,
    "month": 9,
    "year": 2021,
    "day_of_week": 1,  # Tuesday
}

prediction = predictor.predict(rush_scenario)
print(json.dumps(prediction, indent=2))